In [1]:
import json
import re
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional

class TicketWorkflowEngine:
    """
    Task 2 Workflow Engine:
    - Extracted entities & missing detail checks
    - SLA-aware priority calculation (excluding weekends/holidays)
    - Dynamic agent routing based on skill, workload, & availability
    - Duplicate detection and issue grouping/separation
    - Masked handoff summary generation
    """

    def __init__(self, holidays: List[str] = None):
        # Default holiday configuration (YYYY-MM-DD)
        self.holidays = set(holidays or ["2026-01-01", "2026-12-25"])

        # Simulated agent pool
        self.agents = [
            {"id": "AGENT_01", "name": "Alice", "skills": ["payments", "billing"], "workload": 2, "available": True},
            {"id": "AGENT_02", "name": "Bob", "skills": ["technical", "account"], "workload": 5, "available": True},
            {"id": "AGENT_03", "name": "Charlie", "skills": ["payments", "general"], "workload": 1, "available": False},
            {"id": "AGENT_04", "name": "Diana", "skills": ["legal", "high_priority"], "workload": 0, "available": True}
        ]

        # Existing ticket database for duplicate detection
        self.existing_tickets = []

    # ==========================================================
    # 1. ENTITY EXTRACTION & DATA MASKING
    # ==========================================================
    def extract_entities(self, conversation_text: str) -> Dict[str, Any]:
        """Extracts customer, order, product, issue, evidence, and contact details."""
        order_match = re.search(r'\b(ORD-\d{4,8}|\b\d{5,8}\b)', conversation_text)
        email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', conversation_text)
        phone_match = re.search(r'\+?\d{10,12}', conversation_text)

        # Basic issue category tagging
        issue_category = "general"
        if any(w in conversation_text.lower() for w in ["charged", "payment", "refund", "invoice", "double"]):
            issue_category = "payments"
        elif any(w in conversation_text.lower() for w in ["hacked", "login", "password", "compromise"]):
            issue_category = "account"
        elif any(w in conversation_text.lower() for w in ["lawyer", "legal", "sue"]):
            issue_category = "legal"

        extracted = {
            "order_id": order_match.group(0) if order_match else None,
            "contact_email": email_match.group(0) if email_match else None,
            "contact_phone": phone_match.group(0) if phone_match else None,
            "issue_category": issue_category,
            "issue_description": conversation_text
        }

        # Check missing mandatory information
        mandatory_fields = ["order_id", "contact_email"]
        missing_fields = [field for field in mandatory_fields if not extracted[field]]
        extracted["missing_mandatory_info"] = missing_fields

        return extracted

    def generate_masked_summary(self, raw_text: str) -> str:
        """Masks PII such as emails, phone numbers, and payment details for agent handoffs."""
        # Mask emails
        masked = re.sub(r'([\w\.-]+)@([\w\.-]+\.\w+)', r'***@\2', raw_text)
        # Mask phone numbers
        masked = re.sub(r'\+?\d{6,12}', r'******', masked)
        # Mask card-like numbers
        masked = re.sub(r'\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b', '****-****-****-****', masked)
        return masked

    # ==========================================================
    # 2. SLA & PRIORITY CALCULATIONS
    # ==========================================================
    def is_business_day(self, date_obj: datetime) -> bool:
        """Excludes configured weekends (Sat/Sun) and holidays."""
        if date_obj.weekday() >= 5:  # Saturday/Sunday
            return False
        if date_obj.strftime("%Y-%m-%d") in self.holidays:
            return False
        return True

    def calculate_effective_sla_hours(self, start_time: datetime, end_time: datetime) -> float:
        """Calculates elapsed working time excluding weekends and specified holidays."""
        current = start_time
        working_hours = 0.0

        while current < end_time:
            if self.is_business_day(current):
                # Count hours during 9 AM - 5 PM business window
                if 9 <= current.hour < 17:
                    working_hours += 1.0
            current += timedelta(hours=1)

        return working_hours

    def calculate_priority_and_sla(
        self,
        severity: str,
        sentiment: str,
        waiting_time_mins: int,
        customer_impact: str
    ) -> Dict[str, Any]:
        """
        Calculates Priority (P1 to P4) and dynamic SLA hours using input parameters.
        """
        score = 0

        # Severity score
        severity_map = {"critical": 40, "high": 30, "medium": 20, "low": 10}
        score += severity_map.get(severity.lower(), 10)

        # Sentiment multiplier score
        sentiment_map = {"urgent": 25, "frustrated": 20, "negative": 15, "neutral": 5, "positive": 0}
        score += sentiment_map.get(sentiment.lower(), 5)

        # Customer Impact
        impact_map = {"high": 25, "medium": 15, "low": 5}
        score += impact_map.get(customer_impact.lower(), 5)

        # Waiting time weight
        score += min(waiting_time_mins // 5, 10)

        # Priority Level & SLA Allowance Thresholds
        if score >= 75:
            priority = "P1 - Critical"
            allowed_sla_hours = 2.0
        elif score >= 50:
            priority = "P2 - High"
            allowed_sla_hours = 6.0
        elif score >= 30:
            priority = "P3 - Medium"
            allowed_sla_hours = 12.0
        else:
            priority = "P4 - Low"
            allowed_sla_hours = 24.0

        return {
            "priority": priority,
            "calculated_score": score,
            "allowed_sla_hours": allowed_sla_hours
        }

    def check_sla_status(self, created_time: datetime, current_time: datetime, allowed_hours: float) -> Dict[str, Any]:
        """Generates 75% consumable time warnings and identifies breaches."""
        elapsed_hours = self.calculate_effective_sla_hours(created_time, current_time)
        usage_ratio = elapsed_hours / allowed_hours if allowed_hours > 0 else 1.0

        status = "ON_TRACK"
        if usage_ratio >= 1.0:
            status = "BREACHED"
        elif usage_ratio >= 0.75:
            status = "WARNING_75_PERCENT_CONSUMED"

        return {
            "elapsed_working_hours": elapsed_hours,
            "allowed_hours": allowed_hours,
            "usage_ratio": round(usage_ratio, 2),
            "status": status
        }

    # ==========================================================
    # 3. DUPLICATE DETECTION & TICKET GROUPING
    # ==========================================================
    def detect_duplicate_or_group(self, new_entities: Dict[str, Any]) -> Dict[str, Any]:
        """Detects duplicate requests or groups related issues based on Order ID or Contact Email."""
        for existing in self.existing_tickets:
            # Check duplicate
            if (new_entities["order_id"] and new_entities["order_id"] == existing.get("order_id")) and \
               new_entities["issue_category"] == existing.get("issue_category"):
                return {
                    "action": "DUPLICATE_DETECTED",
                    "parent_ticket_id": existing["ticket_id"],
                    "relationship": "Grouped under existing open ticket"
                }

            # Check related issue grouping (same email/customer, different issue)
            if new_entities["contact_email"] and new_entities["contact_email"] == existing.get("contact_email"):
                return {
                    "action": "RELATED_ISSUE_GROUPED",
                    "parent_ticket_id": existing["ticket_id"],
                    "relationship": "Linked as related customer issue"
                }

        return {"action": "CREATE_NEW_TICKET", "parent_ticket_id": None, "relationship": "Standalone Issue"}

    # ==========================================================
    # 4. AGENT ROUTING ENGINE
    # ==========================================================
    def route_ticket(self, issue_category: str, priority: str, current_time: datetime) -> Dict[str, Any]:
        """Routes ticket according to agent skills, availability, workload, and working hours."""

        # Check business hours (Mon-Fri 9 AM - 5 PM)
        is_after_hours = not (self.is_business_day(current_time) and 9 <= current_time.hour < 17)
        if is_after_hours:
            return {
                "assigned_agent": None,
                "queue": "AFTER_HOURS_HOLD_QUEUE",
                "routing_reason": "Ticket created outside business hours. Scheduled for next working day."
            }

        # Filter candidate agents by skills and availability
        candidates = [
            agent for agent in self.agents
            if agent["available"] and issue_category in agent["skills"]
        ]

        # Priority fallback: if high priority or no matching skill, pick any available lowest-workload agent
        if not candidates or "P1" in priority:
            candidates = [agent for agent in self.agents if agent["available"]]

        if not candidates:
            return {
                "assigned_agent": None,
                "queue": "UNASSIGNED_BACKLOG_QUEUE",
                "routing_reason": "No available agents found at this moment."
            }

        # Select agent with lowest current workload
        best_agent = min(candidates, key=lambda x: x["workload"])
        best_agent["workload"] += 1  # Increment workload upon assignment

        return {
            "assigned_agent": best_agent["name"],
            "agent_id": best_agent["id"],
            "queue": f"TIER_1_{issue_category.upper()}_QUEUE",
            "routing_reason": f"Matched skills ({issue_category}) with lowest workload ({best_agent['workload'] - 1} tasks)."
        }

    # ==========================================================
    # 5. PIPELINE EXECUTION ENGINE
    # ==========================================================
    def process_incoming_conversation(
        self,
        text: str,
        severity: str,
        sentiment: str,
        customer_impact: str,
        created_time: datetime,
        current_time: datetime
    ) -> Dict[str, Any]:

        # Step 1: Entity Extraction & Masking
        entities = self.extract_entities(text)
        masked_summary = self.generate_masked_summary(text)

        # Step 2: Check missing mandatory info
        if entities["missing_mandatory_info"]:
            return {
                "status": "INFO_REQUIRED",
                "message": f"Missing mandatory information: {', '.join(entities['missing_mandatory_info'])}. Please prompt customer.",
                "extracted_entities": entities
            }

        # Step 3: Priority & SLA Rules
        sla_prio = self.calculate_priority_and_sla(
            severity, sentiment, waiting_time_mins=10, customer_impact=customer_impact
        )
        sla_check = self.check_sla_status(created_time, current_time, sla_prio["allowed_sla_hours"])

        # Step 4: Duplicate & Grouping Check
        grouping_info = self.detect_duplicate_or_group(entities)

        # Step 5: Routing
        routing_info = self.route_ticket(entities["issue_category"], sla_prio["priority"], current_time)

        # Construct final Structured Ticket Object
        ticket = {
            "ticket_id": f"TICK-{len(self.existing_tickets) + 101}",
            "created_time": created_time.isoformat(),
            "extracted_details": entities,
            "masked_handoff_summary": masked_summary,
            "priority": sla_prio["priority"],
            "sla_info": {
                "allowed_hours": sla_prio["allowed_sla_hours"],
                "sla_status": sla_check["status"],
                "usage_ratio": sla_check["usage_ratio"]
            },
            "grouping_status": grouping_info,
            "routing_assignment": routing_info
        }

        # Store in internal DB for future duplicate checks
        self.existing_tickets.append({
            "ticket_id": ticket["ticket_id"],
            "order_id": entities["order_id"],
            "contact_email": entities["contact_email"],
            "issue_category": entities["issue_category"]
        })

        return ticket


# ==========================================================
# TEST RUN & HIDDEN EVALUATION SIMULATION
# ==========================================================
if __name__ == "__main__":
    engine = TicketWorkflowEngine()

    print("="*60)
    print("RUNNING TASK 2 WORKFLOW ENGINE")
    print("="*60)

    # ---------------------------------------------------------
    # TEST 1: Standard Ticket & Agent Routing (Business Hours)
    # ---------------------------------------------------------
    print("\n--- TEST 1: Normal Ticket Processing ---")
    conv_1 = "Hi, my order ORD-99281 was charged twice ($120). Contact me at john.doe@example.com or +19876543210."
    res_1 = engine.process_incoming_conversation(
        text=conv_1,
        severity="high",
        sentiment="urgent",
        customer_impact="high",
        created_time=datetime(2026, 10, 5, 10, 0), # Mon 10 AM
        current_time=datetime(2026, 10, 5, 11, 0)
    )
    print(json.dumps(res_1, indent=2))

    # ---------------------------------------------------------
    # TEST 2: Duplicate Ticket Handling
    # ---------------------------------------------------------
    print("\n--- TEST 2: Duplicate Ticket Detection ---")
    conv_2 = "Re-sending: order ORD-99281 payment issue. Please fix john.doe@example.com"
    res_2 = engine.process_incoming_conversation(
        text=conv_2,
        severity="high",
        sentiment="frustrated",
        customer_impact="medium",
        created_time=datetime(2026, 10, 5, 11, 30),
        current_time=datetime(2026, 10, 5, 11, 35)
    )
    print(json.dumps(res_2, indent=2))

    # ---------------------------------------------------------
    # TEST 3: After-Hours Routing + Missing Mandatory Info Request
    # ---------------------------------------------------------
    print("\n--- TEST 3: Missing Info Detection ---")
    conv_3 = "My account was hacked! Help me please!"
    res_3 = engine.process_incoming_conversation(
        text=conv_3,
        severity="critical",
        sentiment="urgent",
        customer_impact="high",
        created_time=datetime(2026, 10, 3, 20, 0), # Saturday Night
        current_time=datetime(2026, 10, 3, 20, 10)
    )
    print(json.dumps(res_3, indent=2))

    # ---------------------------------------------------------
    # TEST 4: SLA Consumed Warning (Weekend Excluded)
    # ---------------------------------------------------------
    print("\n--- TEST 4: SLA Warning Excludes Weekend ---")
    conv_4 = "Issue with order ORD-44112. Email user@test.com."
    res_4 = engine.process_incoming_conversation(
        text=conv_4,
        severity="critical",
        sentiment="urgent",
        customer_impact="high",
        created_time=datetime(2026, 10, 2, 16, 0), # Friday 4 PM
        current_time=datetime(2026, 10, 5, 10, 0)  # Monday 10 AM (Only 2 working hrs elapsed)
    )
    print(json.dumps(res_4, indent=2))

RUNNING TASK 2 WORKFLOW ENGINE

--- TEST 1: Normal Ticket Processing ---
{
  "ticket_id": "TICK-101",
  "created_time": "2026-10-05T10:00:00",
  "extracted_details": {
    "order_id": "ORD-99281",
    "contact_email": "john.doe@example.com",
    "contact_phone": "+19876543210",
    "issue_category": "payments",
    "issue_description": "Hi, my order ORD-99281 was charged twice ($120). Contact me at john.doe@example.com or +19876543210.",
    "missing_mandatory_info": []
  },
  "masked_handoff_summary": "Hi, my order ORD-99281 was charged twice ($120). Contact me at ***@example.com or ******.",
  "priority": "P1 - Critical",
  "sla_info": {
    "allowed_hours": 2.0,
    "sla_status": "ON_TRACK",
    "usage_ratio": 0.5
  },
  "grouping_status": {
    "action": "CREATE_NEW_TICKET",
    "parent_ticket_id": null,
    "relationship": "Standalone Issue"
  },
  "routing_assignment": {
    "assigned_agent": "Diana",
    "agent_id": "AGENT_04",
    "queue": "TIER_1_PAYMENTS_QUEUE",
    "routing_